# **Лабораторная работа №1**

In [1]:
from pyspark import SparkContext, SparkConf
from typing import NamedTuple
from datetime import datetime
import math

conf = SparkConf().setAppName("Lab1").setMaster("local[*]")
sc = SparkContext(conf=conf)

print(f"Версия Spark: {sc.version}")

Версия Spark: 4.0.2


In [2]:
# Загрузка и очистка данных
tripData = sc.textFile("trip.csv")
tripsHeader = tripData.first()
tripsRaw = tripData.filter(lambda row: row != tripsHeader).map(lambda row: row.split(",", -1))

stationData = sc.textFile("station.csv")
stationsHeader = stationData.first()
stationsRaw = stationData.filter(lambda row: row != stationsHeader).map(lambda row: row.split(",", -1))

# Определение моделей
class Station(NamedTuple):
    station_id: int
    name: str
    lat: float
    long: float
    dockcount: int
    landmark: str
    installation: datetime

class Trip(NamedTuple):
    trip_id: int
    duration: int
    start_date: datetime
    start_station_name: str
    start_station_id: int
    end_date: datetime
    end_station_name: str
    end_station_id: int
    bike_id: int
    subscription_type: str
    zip_code: str

def initTrip(trips):
    for trip in trips:
        try:
            yield Trip(int(trip[0]), int(trip[1]), datetime.strptime(trip[2], '%m/%d/%Y %H:%M'),
                       trip[3], int(trip[4]), datetime.strptime(trip[5], '%m/%d/%Y %H:%M'),
                       trip[6], int(trip[7]), int(trip[8]), trip[9], trip[10])
        except: pass

def initStation(stations):
    for s in stations:
        yield Station(int(s[0]), s[1], float(s[2]), float(s[3]), int(s[4]), s[5], datetime.strptime(s[6], '%m/%d/%Y'))

# Парсинг и кэширование
trips = tripsRaw.mapPartitions(initTrip).cache()
stations = stationsRaw.mapPartitions(initStation).cache()

print(f"Количество станций: {stations.count()}")
print(f"Количество поездок (trips): {trips.count()}")

Количество станций: 70
Количество поездок (trips): 669959


# **1. Найти велосипед с максимальным временем пробега.**

In [3]:
bike_durations = trips.map(lambda t: (t.bike_id, t.duration)).reduceByKey(lambda a, b: a + b)
max_duration_bike = bike_durations.sortBy(lambda x: x[1], ascending=False).first()
print(f"Велосипед {max_duration_bike[0]} имеет максимальный пробег: {max_duration_bike[1]} сек.")

Велосипед 535 имеет максимальный пробег: 18611693 сек.


# **2. Найти наибольшее геодезическое расстояние между станциями.**

In [4]:
def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    dlat, dlon = math.radians(lat2 - lat1), math.radians(lon2 - lon1)
    a = math.sin(dlat/2)**2 + math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.sin(dlon/2)**2
    return 2 * R * math.asin(math.sqrt(a))

station_pairs = stations.cartesian(stations).filter(lambda x: x[0].station_id < x[1].station_id)
distances = station_pairs.map(lambda x: (f"{x[0].name} - {x[1].name}", haversine(x[0].lat, x[0].long, x[1].lat, x[1].long)))
max_dist = distances.sortBy(lambda x: x[1], ascending=False).first()
print(f"Максимальное расстояние: {max_dist[1]:.2f} км ({max_dist[0]})")

Максимальное расстояние: 69.92 км (SJSU - San Salvador at 9th - Embarcadero at Sansome)


# **3. Найти путь велосипеда с максимальным временем пробега через станции.**

In [5]:
target_bike = max_duration_bike[0]
bike_path = trips.filter(lambda t: t.bike_id == target_bike).sortBy(lambda t: t.start_date)

full_path = bike_path.map(lambda t: (t.start_station_name, t.end_station_name)).collect()

print(f"--- Отчет по велосипеду {target_bike} ---")
print(f"Общее количество остановок: {len(full_path)}")
print("\nПервые 10 станций (перемещений) маршрута:")
for i, (start, end) in enumerate(full_path[:10], 1):
    print(f"{i}. {start} -> {end}")

# print(f"\nПолный маршрут: {full_path}")

--- Отчет по велосипеду 535 ---
Общее количество остановок: 1328

Первые 10 станций (перемещений) маршрута:
1. Post at Kearney -> San Francisco Caltrain (Townsend at 4th)
2. San Francisco Caltrain (Townsend at 4th) -> San Francisco Caltrain 2 (330 Townsend)
3. San Francisco Caltrain 2 (330 Townsend) -> Market at Sansome
4. Market at Sansome -> 2nd at South Park
5. 2nd at Townsend -> Davis at Jackson
6. San Francisco City Hall -> Civic Center BART (7th at Market)
7. Civic Center BART (7th at Market) -> Post at Kearney
8. Post at Kearney -> Embarcadero at Sansome
9. Embarcadero at Sansome -> Washington at Kearney
10. Washington at Kearney -> Market at Sansome


# **4. Найти количество велосипедов в системе.**

In [6]:
bikes_count = trips.map(lambda t: t.bike_id).distinct().count()
print(f"Количество велосипедов в системе: {bikes_count}")

Количество велосипедов в системе: 700


# **5. Найти пользователей потративших на поездки более 3 часов.**

In [7]:
user_times = trips.map(lambda t: (t.zip_code, t.duration)).reduceByKey(lambda a, b: a + b)
heavy_users_rdd = user_times.filter(lambda x: x[1] > 10800).sortBy(lambda x: x[1], ascending=False)

print(f"Пользователей, потративших более 3 часов: {heavy_users_rdd.count()}")
print("\nТоп 10 пользователей по длительности:")
print(f"{'ZIP-код':<15} | {'Время (сек)':<15} | {'Время (часы)':<10}")
print("-" * 45)

for zip_code, duration in heavy_users_rdd.take(10):
    # Если zip_code пустой, выводим 'Unknown'
    label = zip_code if zip_code else "Unknown"
    print(f"{label:<15} | {duration:<15} | {duration/3600:>10.2f}")

sc.stop()

Пользователей, потративших более 3 часов: 3661

Топ 10 пользователей по длительности:
ZIP-код         | Время (сек)     | Время (часы)
---------------------------------------------
94107           | 49757162        |   13821.43
nil             | 45725550        |   12701.54
Unknown         | 27723273        |    7700.91
94105           | 25596128        |    7110.04
94133           | 21637675        |    6010.47
94102           | 19128021        |    5313.34
94103           | 19127388        |    5313.16
95531           | 17270400        |    4797.33
94111           | 14244997        |    3956.94
95112           | 12742370        |    3539.55


In [8]:
sc.stop()